# NutriMenú · Protein Food — RAG con Ollama + Flask

Este notebook prepara y ejecuta el proyecto final a partir de los notebooks del curso que compartiste.

**Arquitectura:** preparación de documentos → embeddings/LanceDB → búsqueda híbrida BM25+densa → guardrails → filtros de kcal → memoria SQLite por `room_id` → interfaz Flask.

> Las kcal de la base son estimadas. No se presentan como información nutricional oficial.


## 0. Ubicarse en la raíz del proyecto

Puedes abrir este notebook desde la carpeta `notebooks/`; la celda encuentra `app.py` y cambia el directorio de trabajo.


In [3]:
from pathlib import Path
import os, sys

ROOT = Path.cwd().resolve()
if not (ROOT / "app.py").exists():
    if (ROOT.parent / "app.py").exists():
        ROOT = ROOT.parent
    else:
        raise FileNotFoundError("No encuentro app.py. Abre el notebook dentro del proyecto NutriMenu_ProteinFood_RAG.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Proyecto:", ROOT)


Proyecto: C:\Users\camil\Downloads\NutriMenu_ProteinFood_RAG\NutriMenu_ProteinFood_RAG


## 1. Instalar dependencias

Ejecuta esta celda una sola vez por entorno.


In [4]:
%pip install -q -r requirements.txt
print("Dependencias listas.")


Note: you may need to restart the kernel to use updated packages.
Dependencias listas.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Verificar Ollama y los modelos

Por defecto el proyecto usa `gemma3:4b` y `embeddinggemma:300m`. Si faltan, usa `ollama pull`.


In [5]:
import requests
from config import OLLAMA_BASE_URL, OLLAMA_LLM_MODEL, OLLAMA_EMBED_MODEL

try:
    r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
    r.raise_for_status()
    modelos = sorted(m["name"] for m in r.json().get("models", []))
    print("Ollama responde en", OLLAMA_BASE_URL)
    print("Modelos locales:")
    for m in modelos: print(" -", m)
except Exception as e:
    print("Ollama no responde:", type(e).__name__, e)
    print("Abre Ollama o ejecuta: ollama serve")

print("\nLLM esperado:", OLLAMA_LLM_MODEL)
print("Embeddings esperados:", OLLAMA_EMBED_MODEL)
print(f"Si faltan: ollama pull {OLLAMA_LLM_MODEL}")
print(f"           ollama pull {OLLAMA_EMBED_MODEL}")


Ollama responde en http://localhost:11434
Modelos locales:
 - embeddinggemma:300m
 - gemma3:4b
 - gemma4:12b
 - mxbai-embed-large:latest
 - nomic-embed-text:latest
 - shieldgemma:2b

LLM esperado: gemma3:4b
Embeddings esperados: embeddinggemma:300m
Si faltan: ollama pull gemma3:4b
           ollama pull embeddinggemma:300m


## 3. Revisar la base de conocimiento y los PNG

El RAG usa JSON derivado del Excel. Cada producto contiene `imagen_png`, que apunta a `static/products/`.


In [6]:
import json
from pathlib import Path

catalog = json.loads(Path("data/catalogo_rag.json").read_text(encoding="utf-8"))
print("Productos:", len(catalog))
print("Ejemplo:")
for k in ["id","producto","precio_pen","kcal_min_est","kcal_max_est","nivel_proteico","imagen_png"]:
    print(f"  {k}: {catalog[0][k]}")

faltantes = [p["imagen_png"] for p in catalog if not (Path("static/products") / p["imagen_png"]).exists()]
print("\nPNG faltantes:", len(faltantes))
if faltantes: print(faltantes[:10])


Productos: 59
Ejemplo:
  id: PROD_001
  producto: Proteincookie Vainilla y Chip
  precio_pen: 12.9
  kcal_min_est: 230
  kcal_max_est: 330
  nivel_proteico: Media
  imagen_png: prod_001_proteincookie_vainilla_y_chip.png

PNG faltantes: 0


## 4. Construir el índice RAG

Se indexan reglas, seguridad, ejemplos y los 59 productos con embeddings de Ollama en LanceDB.


In [7]:
from rag.ingest import build_index, load_documents

docs = load_documents()
print("Chunks a indexar:", len(docs))
vectorstore = build_index(force=True)
print("Índice construido en storage/lancedb")


Chunks a indexar: 119
Índice construido en storage/lancedb


## 5. Probar recuperación híbrida

Combina búsqueda vectorial y BM25 usando Reciprocal Rank Fusion.


In [8]:
from rag.engine import NutriMenuRAG

rag = NutriMenuRAG()
consulta = "Quiero un postre de alrededor de 350 kcal"
recuperados = rag.hybrid_retrieve(consulta, k=6)
for i, d in enumerate(recuperados, 1):
    print(i, d.metadata.get("tipo"), d.metadata.get("producto") or d.metadata.get("titulo"))
    print("  ", d.page_content[:180].replace("\n"," "), "...")


1 caso_uso “Quiero un postre de alrededor de 350 kcal.”
   Usuario: “Quiero un postre de alrededor de 350 kcal.” Respuesta/acción esperada: Comparar Softy Cheesecake de Fresa (280–390), Pie de Limón (290–400), Tiramisú (280–390) y Proteinm ...
2 caso_uso “Quiero algo dulce de máximo 250 kcal.”
   Usuario: “Quiero algo dulce de máximo 250 kcal.” Respuesta/acción esperada: Priorizar Alfajor Dulce de Leche (180–250), Alfajor de Chocomenta (170–250), Bi Sheip (170–240) y mostra ...
3 caso_uso “Me quedan 160 kcal y quiero proteína.”
   Usuario: “Me quedan 160 kcal y quiero proteína.” Respuesta/acción esperada: Recomendar Simple 16 Oz (110–160 kcal est., proteína alta). Como alternativa, Helado Artesanal de Maracu ...
4 rango Medio
   Rango kcal ref.: 351–450. Etiqueta: Medio. Ejemplos del menú: Softy Cream de Cookies&Cream (~360 kcal); PROTEINMOUSSE DE CACAO Y MANJAR (~360 kcal); Proteincuchareable De Pay De Li ...
5 rango Moderado
   Rango kcal ref.: 251–350. Etiqueta: Moderado. Ejemplos d

## 6. Probar memoria por sala (`room_id`)

SQLite conserva estado e historial por sala. La prueba crea dos salas distintas.


In [9]:
from rag.memory import create_room, get_state, update_state

room_a = create_room("Prueba 500 kcal")
room_b = create_room("Prueba 250 kcal")
update_state(room_a["id"], {"meal_kcal": 500, "flavor": "Salado", "wants_protein": True})
update_state(room_b["id"], {"meal_kcal": 250, "flavor": "Dulce"})

print("Sala A:", room_a["id"], get_state(room_a["id"]))
print("Sala B:", room_b["id"], get_state(room_b["id"]))
assert get_state(room_a["id"])["meal_kcal"] == 500
assert get_state(room_b["id"])["meal_kcal"] == 250
print("\nMemoria aislada por room_id: OK")


Sala A: bde1f8aa-fe61-440a-99ab-3107de2a0dcc {'room_id': 'bde1f8aa-fe61-440a-99ab-3107de2a0dcc', 'meal_kcal': 500.0, 'daily_kcal': None, 'kcal_mode': 'strict', 'flavor': 'Salado', 'consumption_type': None, 'wants_protein': True, 'price_max': None, 'last_intent': None, 'updated_at': '2026-08-15T00:52:02.948564+00:00', 'weight_kg': None, 'height_cm': None, 'age': None, 'formula_sex': None, 'activity_level': None, 'exclusions': []}
Sala B: a4d2466a-121c-4903-9d22-d2e9bff3364e {'room_id': 'a4d2466a-121c-4903-9d22-d2e9bff3364e', 'meal_kcal': 250.0, 'daily_kcal': None, 'kcal_mode': 'strict', 'flavor': 'Dulce', 'consumption_type': None, 'wants_protein': False, 'price_max': None, 'last_intent': None, 'updated_at': '2026-08-15T00:52:02.964445+00:00', 'weight_kg': None, 'height_cm': None, 'age': None, 'formula_sex': None, 'activity_level': None, 'exclusions': []}

Memoria aislada por room_id: OK


## 7. Consulta completa al chatbot

Esta celda sí llama a Gemma mediante Ollama.


In [10]:
from rag.memory import create_room, save_message

room = create_room("Demo NutriMenú")
question = "Tengo 500 kcal para almorzar y quiero algo salado y proteico"
save_message(room["id"], "user", question)
result = rag.answer(room["id"], question)
print(result["answer"])
print("\nProductos de las tarjetas:")
for p in result["products"]:
    print(" -", p["producto"], p["kcal_min_est"], "–", p["kcal_max_est"], "kcal", p["imagen_png"])


¡Hola! Claro que sí. Con 500 kcal para almorzar y buscando algo salado y proteico, te recomiendo nuestro Protein Lover. Es una ensalada deliciosa con pollo, espinaca, huevo duro y arroz integral, ideal para saciar tu apetito y obtener proteína. Tiene un rango de calorías estimado entre 380 y 500. ¿Te gustaría que te cuente más detalles sobre este plato?

Productos de las tarjetas:
 - Protein Lover 380 – 500 kcal prod_049_protein_lover.png


## 8. Iniciar la interfaz Flask

La siguiente celda inicia Flask en segundo plano. Luego abre `http://127.0.0.1:5000`.

Si ya hay un servidor usando el puerto 5000, no la ejecutes otra vez.


In [11]:
import subprocess, sys, time

server = subprocess.Popen([sys.executable, "app.py"])
time.sleep(2)
print("Flask iniciado. PID:", server.pid)
print("Abre: http://127.0.0.1:5000")
print("Para detenerlo desde este notebook: server.terminate()")


Flask iniciado. PID: 16560
Abre: http://127.0.0.1:5000
Para detenerlo desde este notebook: server.terminate()


In [12]:
#   0ver.terminate()

## Notas finales

- Las fotografías reales no fueron proporcionadas. Los PNG incluidos son placeholders reemplazables 1:1 manteniendo el nombre.
- Si cambias `OLLAMA_EMBED_MODEL`, reconstruye el índice.
- El historial de cada sala se guarda en `storage/chats.db`.
- El Excel original estructurado está en `data/base_conocimiento_rag_menu.xlsx`.
